# 🛠️ Tutorial 1 (T1): Data Preprocessing in Python
### Machine Learning for Precision Agriculture (Crop & Fertilizer Datasets)

**Goal:** Learn the foundational steps of data preprocessing required before training any machine learning model:
1. **Data Inspection & Cleaning**: Checking dimensions, column types, and handling missing values.
2. **Categorical Encoding**: Converting string columns (like Soil Type & Fertilizer Name) into numbers using `LabelEncoder`.
3. **Feature Scaling**: Normalizing feature magnitudes with `StandardScaler` and `MinMaxScaler`.
4. **Train-Test Splitting**: Partitioning data (80% Train, 20% Test) with `stratify=y` to prevent data leakage.

---
## ⚙️ Step 0: Import Libraries & Setup Data (For Google Colab & Local)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder

# Set plot styling
sns.set_theme(style="whitegrid")
plt.rcParams.update({'figure.dpi': 120})

# Ensure dataset directory exists
os.makedirs("data", exist_ok=True)
crop_csv = os.path.join("data", "Crop_recommendation.csv")
fert_csv = os.path.join("data", "Fertilizer_Prediction.csv")

# Auto-generate demo datasets if not found (for Colab one-click run)
if not os.path.exists(crop_csv):
    print("Creating sample Crop_recommendation.csv...")
    crops = ["rice", "maize", "chickpea", "kidneybeans", "pigeonpeas", "mothbeans", "mungbean", "blackgram", "lentil", "pomegranate"]
    df_c = pd.DataFrame({
        'N': np.random.randint(10, 140, 200),
        'P': np.random.randint(5, 145, 200),
        'K': np.random.randint(15, 205, 200),
        'temperature': np.random.uniform(8.0, 43.0, 200),
        'humidity': np.random.uniform(14.0, 100.0, 200),
        'ph': np.random.uniform(3.5, 9.9, 200),
        'rainfall': np.random.uniform(20.0, 300.0, 200),
        'label': np.random.choice(crops, 200)
    })
    df_c.to_csv(crop_csv, index=False)

if not os.path.exists(fert_csv):
    print("Creating sample Fertilizer_Prediction.csv...")
    ferts = ["Urea", "DAP", "14-35-14", "28-28", "17-17-17", "20-20", "10-26-26"]
    soils = ["Clayey", "Sandy", "Loamy", "Black", "Red"]
    crop_types = ["Maize", "Sugarcane", "Cotton", "Tobacco", "Paddy", "Barley", "Wheat", "Millets"]
    df_f = pd.DataFrame({
        'Temperature': np.random.randint(25, 40, 200),
        'Humidity': np.random.randint(50, 75, 200),
        'Moisture': np.random.randint(25, 70, 200),
        'Soil Type': np.random.choice(soils, 200),
        'Crop Type': np.random.choice(crop_types, 200),
        'Nitrogen': np.random.randint(0, 45, 200),
        'Phosphorous': np.random.randint(0, 45, 200),
        'Potassium': np.random.randint(0, 45, 200),
        'Fertilizer Name': np.random.choice(ferts, 200)
    })
    df_f.to_csv(fert_csv, index=False)

print("Environment and datasets ready!")

---
## 🔍 Step 1: Data Inspection & Missing Value Handling

In [ ]:
# Load datasets
df_crop = pd.read_csv(crop_csv)
df_fert = pd.read_csv(fert_csv)

print("Crop Dataset Dimensions:", df_crop.shape)
print("Fertilizer Dataset Dimensions:", df_fert.shape)

print("\n--- Crop Dataset Missing Values ---")
print(df_crop.isnull().sum())

print("\n--- Fertilizer Dataset Missing Values ---")
print(df_fert.isnull().sum())

display(df_crop.head())

---
## 🏷️ Step 2: Categorical Encoding (Text → Numbers)
Machine learning algorithms cannot perform linear algebra ($X \times W$) on raw text strings like `"Clayey"` or `"Urea"`. We use `LabelEncoder` to convert each category into unique integers.

In [ ]:
df_fert_processed = df_fert.copy()

le_soil = LabelEncoder()
le_crop = LabelEncoder()
le_fert = LabelEncoder()

df_fert_processed['Soil Type_encoded'] = le_soil.fit_transform(df_fert_processed['Soil Type'])
df_fert_processed['Crop Type_encoded'] = le_crop.fit_transform(df_fert_processed['Crop Type'])
df_fert_processed['Fertilizer_encoded'] = le_fert.fit_transform(df_fert_processed['Fertilizer Name'])

print("Soil Type Mapping:", dict(zip(le_soil.classes_, le_soil.transform(le_soil.classes_))))
print("Fertilizer Mapping:", dict(zip(le_fert.classes_, le_fert.transform(le_fert.classes_))))

display(df_fert_processed[['Soil Type', 'Soil Type_encoded', 'Fertilizer Name', 'Fertilizer_encoded']].head())

---
## ⚖️ Step 3: Feature Scaling (StandardScaler vs MinMaxScaler)
Features like Nitrogen ($0-140$) have vastly larger magnitudes than pH ($3.5-9.9$). Scaling brings all features onto an equal playing field so no single column unfairly dominates distance calculations.

In [ ]:
numeric_cols = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
X_raw = df_crop[numeric_cols]

# 1. StandardScaler: Mean = 0, Std = 1
scaler_std = StandardScaler()
X_scaled = scaler_std.fit_transform(X_raw)

# 2. MinMaxScaler: Range = [0, 1]
scaler_minmax = MinMaxScaler()
X_minmax = scaler_minmax.fit_transform(X_raw)

# Compare original vs scaled distributions for Rainfall
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.histplot(df_crop['rainfall'], kde=True, ax=axes[0], color='royalblue')
axes[0].set_title('Original Rainfall (20 - 300 mm)', fontweight='bold')

sns.histplot(X_scaled[:, 6], kde=True, ax=axes[1], color='darkorange')
axes[1].set_title('StandardScaler (Mean=0, Std=1)', fontweight='bold')

sns.histplot(X_minmax[:, 6], kde=True, ax=axes[2], color='forestgreen')
axes[2].set_title('MinMaxScaler (Range: 0 to 1)', fontweight='bold')

plt.tight_layout()
plt.show()

---
## ✂️ Step 4: Train-Test Splitting (80% Train, 20% Test)
We split data using `stratify=y` to ensure that every crop category is represented in identical proportions in both training and testing subsets.

In [ ]:
X = X_scaled
y = df_crop['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"Total Samples:    {X.shape[0]}")
print(f"Training Samples: {X_train.shape[0]} (80%)")
print(f"Testing Samples:  {X_test.shape[0]} (20%)")
print(f"Feature Count:    {X_train.shape[1]}")

print("\nClass distribution in Test Set:")
print(y_test.value_counts())

---
## 🎯 Summary of Key Takeaways
1. **Inspection**: Always verify dataset shape and check for missing values using `df.isnull().sum()`.
2. **Encoding**: Use `LabelEncoder` to convert strings to numbers so algorithms can calculate weights.
3. **Scaling**: Fit `StandardScaler` only on training data to prevent data leakage.
4. **Splitting**: Use `stratify=y` in `train_test_split` to maintain balanced class proportions.